# MUBA Daily Story — Kaggle T4 ×2 Runtime
Open-source Qwen Image Edit runtime for one daily four-frame MUBA story. No paid image API.

In [ ]:
import os, subprocess, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
os.environ['TQDM_DISABLE']='1'
subprocess.check_call([sys.executable,'-m','pip','install','-q','-U','diffusers','transformers','accelerate','sentencepiece','safetensors','huggingface_hub','bitsandbytes'])
import torch
print('CUDA available:',torch.cuda.is_available())
print('GPU count:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i,torch.cuda.get_device_name(i),round(torch.cuda.get_device_properties(i).total_memory/1024**3,1),'GB')
assert torch.cuda.device_count()>=2,'Kaggle T4 x2 accelerator is required'


In [ ]:
import gc, os, torch
from diffusers import QwenImageEditPlusPipeline

# Clear allocations left by any earlier failed loader in this notebook session.
for name in ('pipe', 'pipeline', 'previous', 'output'):
    if name in globals():
        del globals()[name]
gc.collect()
for device in range(torch.cuda.device_count()):
    with torch.cuda.device(device):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
MODEL='seochan99/Qwen-Image-Edit-2511-bnb-nf4'
print('Loading MUBA Daily Story NF4 runtime with CPU offload...')
pipe=QwenImageEditPlusPipeline.from_pretrained(MODEL,torch_dtype=torch.bfloat16,low_cpu_mem_usage=True)
pipe.enable_model_cpu_offload(gpu_id=0)
pipe.set_progress_bar_config(disable=True)
print('MUBA DAILY STORY RUNTIME READY')
print('Model:',MODEL)
